In [0]:
# ==========================================
# PREMIER LEAGUE - BRONZE INGESTION PIPELINE
# ==========================================

import os
import re

VOLUME_PATH = "/Volumes/workspace/default/football_raw/"

DATASET_NAMES = [
    "teams",
    "matches",
    "standings",
    "scorers",
    "competition"
]

# ==========================================
# STEP 1: DISCOVER EXTRACTION BATCHES
# ==========================================

files = os.listdir(VOLUME_PATH)

batches = {}

for filename in files:

    match = re.fullmatch(
        r"(teams|matches|standings|scorers|competition)_(\d{8}_\d{6})\.json",
        filename
    )

    if match:
        dataset_name = match.group(1)
        timestamp = match.group(2)

        if timestamp not in batches:
            batches[timestamp] = {}

        batches[timestamp][dataset_name] = filename


# ==========================================
# STEP 2: FIND THE LATEST COMPLETE BATCH
# ==========================================

complete_batches = {
    timestamp: datasets
    for timestamp, datasets in batches.items()
    if all(name in datasets for name in DATASET_NAMES)
}

if not complete_batches:
    raise ValueError("No complete extraction batch found!")

latest_timestamp = max(complete_batches)

DATASETS = complete_batches[latest_timestamp]

print(f"Selected extraction batch: {latest_timestamp}")


# ==========================================
# STEP 3: INGEST ALL FIVE DATASETS
# ==========================================

for dataset_name in DATASET_NAMES:

    filename = DATASETS[dataset_name]

    print(f"\nProcessing {dataset_name}...")

    file_path = os.path.join(VOLUME_PATH, filename)

    raw_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    table_name = f"workspace.default.bronze_{dataset_name}"

    (
        raw_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Successfully created: {table_name}")
    print(f"Top-level records: {raw_df.count()}")


print("\nBronze ingestion pipeline completed!")